In [99]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    roc_auc_score, average_precision_score, classification_report,
    roc_curve, precision_recall_curve
)
import matplotlib.pyplot as plt
from pathlib import Path
import warnings
import json
import pickle

warnings.filterwarnings('ignore')
torch.manual_seed(42)
np.random.seed(42)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponible: {torch.cuda.is_available()}')

PyTorch: 2.11.0+cu130
CUDA disponible: True


In [ ]:
BASE_DATA_DIR = '../data'
OUTPUT_DIR    = '../data/processed'
MODELS_DIR    = '../models'
RESULTS_DIR   = '../results'

HOURS_BEFORE  = 48
MIN_HOURS     = 6  # Revertido: estancias cortas (3h) añaden ruido, no señal

# Hiperparámetros
BATCH_SIZE    = 64
EPOCHS        = 100
LR            = 5e-4
WEIGHT_DECAY  = 1e-4
# HIDDEN_SIZE: dimensión del estado oculto por dirección del LSTM bidireccional.
# Con bidireccional, la salida efectiva es hidden_size * 2 por timestep.
# La capa de atención produce un vector de contexto de tamaño hidden_size * 2.
HIDDEN_SIZE   = 64
NUM_LAYERS    = 1
DROPOUT       = 0.3
PATIENCE      = 12

RANDOM_STATE_SEED = 42

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {DEVICE}')

Path(MODELS_DIR).mkdir(parents=True, exist_ok=True)
Path(RESULTS_DIR).mkdir(parents=True, exist_ok=True)

In [101]:
features_all = pd.read_parquet(f'{OUTPUT_DIR}/features_all.parquet')
cohort_all   = pd.read_parquet(f'{OUTPUT_DIR}/cohort_all.parquet')

# Temperature Celsius: 99.8% nulos en NB1 → eliminada
FEATURE_COLS = [
    c for c in features_all.columns
    if c not in ['stay_id', 'hour_bucket', 'Temperature Celsius']
]
N_FEATURES = len(FEATURE_COLS)

print(f'Estancias totales: {cohort_all["stay_id"].nunique():,}')
print(f'  Sepsis  (label=1): {(cohort_all["label"]==1).sum():,}')
print(f'  Control (label=0): {(cohort_all["label"]==0).sum():,}')
print(f'Features ({N_FEATURES}): {FEATURE_COLS}')

Estancias totales: 59,932
  Sepsis  (label=1): 29,966
  Control (label=0): 29,966
Features (16): ['Arterial Blood Pressure mean', 'GCS - Motor Response', 'GCS - Verbal Response', 'Heart Rate', 'Non Invasive Blood Pressure diastolic', 'Non Invasive Blood Pressure systolic', 'O2 saturation pulseoxymetry', 'PEEP set', 'Respiratory Rate', 'Bilirubin, Total', 'Creatinine', 'Lactate', 'Platelet Count', 'Urea Nitrogen', 'pH', 'pO2']


In [102]:
# Agrupamos los datos por estancia y contamos las secuencias horarias que tenemos (cuantas horas tiene cada estancia)
stays_coverage = features_all.groupby('stay_id')['hour_bucket'].count()
# Filtramos por el minimo de horas establecido previamente y obtenemos una lista de identificadores
valid_stay_ids = stays_coverage[stays_coverage >= MIN_HOURS].index

# Extraemos etiquetas y pacientes a partir de las estancias filtradas
stay_labels_v = cohort_all[cohort_all['stay_id'].isin(valid_stay_ids)][
    ['stay_id', 'label', 'subject_id']
].reset_index(drop=True)

# Obtenemos listas para poder usar GroupShuffleSplit
all_stays = stay_labels_v['stay_id'].values # Lista de identificadores de estancias
all_labels = stay_labels_v['label'].values # Sepsis o control (sepsis / no sepsis) [0, 1, 1, 0...]
all_subjects = stay_labels_v['subject_id'].values # Lista de identificadores de pacientes

# Split 1: train + val (80%) / test (20%)
gss1 = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE_SEED)
tv_idx, test_idx = next(gss1.split(all_stays, all_labels, groups=all_subjects))

# Split 2: train (75%) / val (25%) del 80% anterior → 60/20/20 global
gss2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_STATE_SEED)
tr_idx, val_idx = next(gss2.split(
    all_stays[tv_idx], all_labels[tv_idx], groups=all_subjects[tv_idx]
))

# Extraemos los identificadores de estancia
train_stays = all_stays[tv_idx][tr_idx] # 80% → 75% de ese 80% = 60% global
val_stays = all_stays[tv_idx][val_idx] # 80% → 25% de ese 80% = 20% global
test_stays = all_stays[test_idx] # 20% global

# Creamos un mapa de estancia → paciente y se usa para obtener que pacientes hay en cada conjunto.
s2sub = cohort_all.set_index('stay_id')['subject_id']
tr_sub = set(s2sub[s2sub.index.isin(train_stays)]) # pacientes en train
va_sub = set(s2sub[s2sub.index.isin(val_stays)]) # pacientes en val
te_sub = set(s2sub[s2sub.index.isin(test_stays)]) # pacientes en test

print(f'Train: {len(train_stays):,} estancias | {len(tr_sub):,} pacientes')
print(f'Val: {len(val_stays):,} estancias | {len(va_sub):,} pacientes')
print(f'Test: {len(test_stays):,} estancias | {len(te_sub):,} pacientes')

Train: 9,589 estancias | 7,803 pacientes
Val: 3,224 estancias | 2,601 pacientes
Test: 3,207 estancias | 2,602 pacientes


In [103]:
def build_tensor(features_df, stay_ids, feature_cols, hours=HOURS_BEFORE):
    """Convierte features_df en tensor numpy (n, hours, n_features)."""

    stay_ids = list(stay_ids) # Lista de identificadores de estancias

    n = len(stay_ids) # Numero de estancias
    F = len(feature_cols) # Numero de variables
    X = np.full((n, hours, F), np.nan, dtype=np.float32) # Matriz rellena con NaN con forma (estancias, 48 horas, N variables)
    
    # Asignamos a cada estancia su posicion en la matriz
    id_to_idx = pd.Series(np.arange(n), index=stay_ids) # Secuencia de posiciones, establecemos como indice el identificador de estancia
    col_to_idx = {c: j for j, c in enumerate(feature_cols)} # Crea una secuencia de posiciones para las variables

    # Filtramos solo las filas relevantes (solo las secuencias dentro de la ventana de 48h)
    df = features_df[
        features_df['stay_id'].isin(set(stay_ids)) &
        features_df['hour_bucket'].between(-hours, -1)
    ].copy()
    
    df['_i'] = id_to_idx[df['stay_id'].values].values # Añadir la posición de cada estancia en la matriz
    
    # Convertir hour_bucket de negativo a positivo: -48 → 0, -1 → 47
    # para que sirva como índice directo en el eje temporal del cubo
    df['_t'] = (df['hour_bucket'] + hours).astype(int)

    df = df.dropna(subset=['_i']) # Eliminamos filas cuyo stay_id no tiene posición asignada (no deberían existir)

    # Rellenar la matriz variable a variable
    for col, j in col_to_idx.items():
        if col not in df.columns: # Saltar si la variable no está en los datos
            continue
        
        # Extraemos solo las filas sin valores nulos
        sub = df[['_i', '_t', col]].dropna(subset=[col])

        i_idx = sub['_i'].astype(int).values # Índice de estancia
        t_idx = sub['_t'].astype(int).values # Índice de hora
        vals = sub[col].values.astype(np.float32) # Valor clínico
        
        # Filtro de seguridad: descartar índices fuera del rango válido [0, 47]
        mask = (t_idx >= 0) & (t_idx < hours)

        # Colocar cada valor en su celda exacta de la matriz [estancia, hora, variable]
        X[i_idx[mask], t_idx[mask], j] = vals[mask]

    return X

# Construir un tensor para cada conjunto (entrenamiento, validacion y test)
X_train_raw = build_tensor(features_all, train_stays, FEATURE_COLS)
X_val_raw = build_tensor(features_all, val_stays, FEATURE_COLS)
X_test_raw = build_tensor(features_all, test_stays, FEATURE_COLS)

# Extraemos las etiquetas (0/1) para cada conjunto usando el stay_id como identificador
label_map = stay_labels_v.set_index('stay_id')['label']
y_train = label_map[train_stays].values.astype(np.float32)
y_val = label_map[val_stays].values.astype(np.float32)
y_test = label_map[test_stays].values.astype(np.float32)

print(f'X_train: {X_train_raw.shape} | positivos: {y_train.mean():.2%}')
print(f'X_val: {X_val_raw.shape} | positivos: {y_val.mean():.2%}')
print(f'X_test: {X_test_raw.shape} | positivos: {y_test.mean():.2%}')

X_train: (9589, 48, 16) | positivos: 56.35%
X_val: (3224, 48, 16) | positivos: 58.00%
X_test: (3207, 48, 16) | positivos: 57.59%


In [104]:
def apply_forward_fill_vectorized(X):
    """
    X : (n, T, F) con NaN.
    Convierte cada secuencia (T, F) a DataFrame, aplica ffill y reconstituye.
    """
    n, T, F = X.shape
    X_ff = X.copy()

    for i in range(n):
        df_tmp = pd.DataFrame(X[i])
        df_ff  = df_tmp.ffill(axis=0)
        X_ff[i] = df_ff.values

    return X_ff

# Calcular máscaras ANTES del forward fill (desde los NaN originales)
# La máscara debe reflejar mediciones reales, no valores propagados
M_train = (~np.isnan(X_train_raw)).astype(np.float32)
M_val   = (~np.isnan(X_val_raw)).astype(np.float32)
M_test  = (~np.isnan(X_test_raw)).astype(np.float32)

X_train_ff = apply_forward_fill_vectorized(X_train_raw)
X_val_ff   = apply_forward_fill_vectorized(X_val_raw)
X_test_ff  = apply_forward_fill_vectorized(X_test_raw)

pct_nan_antes   = np.isnan(X_train_raw).mean()
pct_nan_despues = np.isnan(X_train_ff).mean()
print(f'NaN antes del ffill:   {pct_nan_antes:.1%}')
print(f'NaN después del ffill: {pct_nan_despues:.1%}')

NaN antes del ffill:   98.0%
NaN después del ffill: 83.2%


In [105]:
def apply_missingness_mask(X, train_medians=None, external_mask=None):
    """
    Estrategia de imputación con missingness mask:
    1. Construye máscara binaria M: 1 = valor real, 0 = imputado.
    2. Rellena NaN con la mediana global del training (valor neutro).
    3. Concatena [X_imputed | M] → shape (n, T, 2*F).
    La máscara preserva la señal de ausencia como feature explícito.

    external_mask: máscara pre-computada desde el tensor original con NaN.
    Pasar siempre que X haya sido modificado antes (e.g. forward fill),
    para que la máscara refleje mediciones reales y no valores propagados.
    """

    _, _, F = X.shape

    # Usar máscara externa si se proporciona; si no, calcularla desde X
    M = external_mask if external_mask is not None else (~np.isnan(X)).astype(np.float32)

    X_imp = X.copy()

    if train_medians is None:
        flat = X_imp.reshape(-1, F)
        train_medians = np.nanmedian(flat, axis=0)
        train_medians = np.where(np.isnan(train_medians), 0.0, train_medians)

    for j in range(F):
        mask_nan = np.isnan(X_imp[:, :, j])
        X_imp[:, :, j][mask_nan] = train_medians[j]

    X_out = np.concatenate([X_imp, M], axis=2).astype(np.float32)
    return X_out, train_medians

# Pasar la máscara original (calculada antes del ffill) para no corromper la señal de ausencia
X_train_masked, train_medians = apply_missingness_mask(X_train_ff, external_mask=M_train)
X_val_masked,  _ = apply_missingness_mask(X_val_ff,  train_medians, external_mask=M_val)
X_test_masked, _ = apply_missingness_mask(X_test_ff, train_medians, external_mask=M_test)

F = len(FEATURE_COLS)
print(f'Shape tras aplicar mask: {X_train_masked.shape} → input_size al LSTM: {X_train_masked.shape[2]}')

Shape tras aplicar mask: (9589, 48, 32) → input_size al LSTM: 32


In [106]:
# Solo se normalizan los 16 valores clínicos — las 16 máscaras ya son binarias {0,1} y no deben normalizarse
n_tr, T, F2 = X_train_masked.shape
F = F2 // 2  # Los primeros 16 canales son valores, los últimos 16 son máscaras

scaler = StandardScaler()  # Normalización z-score: media 0, desviación estándar 1

def scale_masked(X, scaler, F, fit=False):
    n, T, _ = X.shape
    X_vals = X[:, :, :F].reshape(-1, F)  # Extraer las 16 variables
    X_mask = X[:, :, F:] # Separar las 16 máscaras

    if fit:
        # En train: calcular media y desviación estándar y aplicar la normalización
        X_vals_scaled = scaler.fit_transform(X_vals)
    else:
        # En val y test: aplicar la normalización con los parámetros calculados en train
        # (evita data leakage: val/test no influyen en el cálculo de la normalización)
        X_vals_scaled = scaler.transform(X_vals)

    X_vals_scaled = X_vals_scaled.reshape(n, T, F).astype(np.float32)  # Restaurar forma 3D

    # Volver a concatenar valores normalizados con las máscaras sin tocar
    return np.concatenate([X_vals_scaled, X_mask], axis=2).astype(np.float32)

# Aplicar normalización: fit solo en train, transform en val y test
X_train_norm = scale_masked(X_train_masked, scaler, F, fit=True)
X_val_norm   = scale_masked(X_val_masked,   scaler, F, fit=False)
X_test_norm  = scale_masked(X_test_masked,  scaler, F, fit=False)

N_INPUT = X_train_norm.shape[2]  # 32 = 16 valores normalizados + 16 máscaras binarias
print(f'Input con mascaras binarias al LSTM: {N_INPUT}  (shape: {X_train_norm.shape})')
print(f'Máscaras — rango: [{X_train_norm[:,:,F:].min():.0f}, {X_train_norm[:,:,F:].max():.0f}]')

Input con mascaras binarias al LSTM: 32  (shape: (9589, 48, 32))
Máscaras — rango: [0, 1]


In [107]:
class SepsisDataset(Dataset):
    """Clase que alimenta al modelo durante el entrenamiento"""
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_ds = SepsisDataset(X_train_norm, y_train)
val_ds   = SepsisDataset(X_val_norm,   y_val)
test_ds  = SepsisDataset(X_test_norm,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

n_pos = int(y_train.sum())
n_neg = len(y_train) - n_pos

N_INPUT = X_train_norm.shape[2]
print(f'Input shape: {X_train_norm.shape} → N_INPUT: {N_INPUT}')
print(f'Train batches: {len(train_loader)} × {BATCH_SIZE}')

Input shape: (9589, 48, 32) → N_INPUT: 32
Train batches: 149 × 64


In [ ]:
class LSTMImproved(nn.Module):
    """BiLSTM con mecanismo de atención sobre el eje temporal.

    Reemplaza el avg+max pooling fijo con atención aprendida:
    el modelo aprende a ponderar cada timestep según su relevancia para la predicción.
    La atención es especialmente útil en series clínicas donde las horas más cercanas
    al momento de predicción suelen ser más informativas.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        lstm_out_size = hidden_size * 2  # bidireccional: concat forward + backward

        # Atención sobre el eje temporal: una puntuación escalar por timestep
        self.attn = nn.Linear(lstm_out_size, 1)

        self.bn1     = nn.BatchNorm1d(lstm_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(lstm_out_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.lstm(x)                                # (batch, T, hidden*2)

        # Atención: softmax sobre la dimensión temporal → suma ponderada
        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


model = LSTMImproved(input_size=N_INPUT).to(DEVICE)
print(model)

In [ ]:
def train_model(model, model_name, pos_weight_value):
    pos_weight = torch.tensor([pos_weight_value], dtype=torch.float32).to(DEVICE)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

    # Scheduler usa val_loss (más estable para controlar el LR)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', patience=3, factor=0.5
    )

    history          = {'train_loss': [], 'val_loss': [], 'val_auroc': []}
    best_val_auroc   = 0.0           # Early stopping sobre AUROC
    patience_counter = 0
    best_path        = f'{MODELS_DIR}/{model_name}.pt'

    for epoch in range(1, EPOCHS + 1):

        # ── Fase entrenamiento ──
        model.train()
        train_loss    = 0.0
        train_samples = 0

        for X_b, y_b in train_loader:
            X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(X_b), y_b)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            train_loss    += loss.item() * len(y_b)
            train_samples += len(y_b)

        train_loss /= train_samples

        # ── Fase validación ──
        model.eval()
        val_loss  = 0.0
        val_probs, val_true = [], []

        with torch.no_grad():
            for X_b, y_b in val_loader:
                X_b, y_b = X_b.to(DEVICE), y_b.to(DEVICE)
                logits = model(X_b)
                val_loss += criterion(logits, y_b).item() * len(y_b)
                val_probs.extend(torch.sigmoid(logits).cpu().numpy())
                val_true.extend(y_b.cpu().numpy())

        val_loss  /= len(val_loader.dataset)
        val_auroc  = roc_auc_score(val_true, val_probs) if len(set(val_true)) > 1 else 0.0

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_auroc'].append(val_auroc)

        # Scheduler usa val_loss
        old_lr = optimizer.param_groups[0]['lr']
        scheduler.step(val_loss)
        new_lr = optimizer.param_groups[0]['lr']
        if new_lr < old_lr:
            print(f'--- Tasa de aprendizaje reducida a: {new_lr:.2e} ---')

        # Early stopping usa val_auroc — guarda el checkpoint con mejor AUROC
        if val_auroc > best_val_auroc:
            best_val_auroc   = val_auroc
            patience_counter = 0
            torch.save(model.state_dict(), best_path)
        else:
            patience_counter += 1

        print(f'Epoch {epoch:3d}/{EPOCHS} | '
              f'Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | '
              f'Val AUROC: {val_auroc:.4f} | LR: {new_lr:.2e} | P: {patience_counter}')

        if patience_counter >= PATIENCE:
            print(f'Early stopping en epoch {epoch}. AUROC no mejoró en {PATIENCE} épocas.')
            break

    print(f'\nMejor val_auroc: {best_val_auroc:.4f}')
    return {
        'history': history,
        'path': best_path,
        'model': model,
        'pos_weight': pos_weight.item()
    }

models = {}

model_name = 'best_lstm_attention'
models[model_name] = train_model(model, model_name, n_neg / max(n_pos, 1))

In [ ]:
class GRUModel(nn.Module):
    """GRU bidireccional con atención — arquitectura equivalente a LSTMImproved.

    GRU vs LSTM: combina las puertas de actualización y reset en una sola celda,
    sin estado de celda separado. Menos parámetros (~75% del LSTM equivalente),
    entrena más rápido y generaliza mejor con datasets pequeños.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
        )

        gru_out_size = hidden_size * 2  # bidireccional

        self.attn    = nn.Linear(gru_out_size, 1)
        self.bn1     = nn.BatchNorm1d(gru_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(gru_out_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.gru(x)                                 # (batch, T, hidden*2)

        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


gru_model = GRUModel(input_size=N_INPUT).to(DEVICE)
print(gru_model)

model_name_gru = 'best_gru_attention'
models[model_name_gru] = train_model(gru_model, model_name_gru, n_neg / max(n_pos, 1))

In [ ]:
class RNNModel(nn.Module):
    """RNN bidireccional con atención — arquitectura equivalente a GRUModel y LSTMImproved.

    RNN vanilla vs GRU/LSTM: sin puertas de actualización ni estado de celda.
    Menos parámetros y más rápido, pero más susceptible al problema del gradiente
    que desaparece en secuencias largas. Sirve como baseline arquitectónico
    para cuantificar cuánto aportan las puertas recurrentes.
    """

    def __init__(
        self,
        input_size: int,
        hidden_size: int = HIDDEN_SIZE,
        num_layers: int = NUM_LAYERS,
        dropout: float = DROPOUT
    ):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
            bidirectional=True,
            nonlinearity='tanh',
        )

        rnn_out_size = hidden_size * 2  # bidireccional

        self.attn    = nn.Linear(rnn_out_size, 1)
        self.bn1     = nn.BatchNorm1d(rnn_out_size)
        self.dropout = nn.Dropout(dropout)
        self.fc1     = nn.Linear(rnn_out_size, 64)
        self.relu    = nn.ReLU()
        self.fc2     = nn.Linear(64, 1)

    def forward(self, x):
        out, _ = self.rnn(x)                                 # (batch, T, hidden*2)

        attn_weights = torch.softmax(self.attn(out), dim=1)  # (batch, T, 1)
        context = (attn_weights * out).sum(dim=1)             # (batch, hidden*2)

        context = self.bn1(context)
        context = self.dropout(context)
        context = self.relu(self.fc1(context))
        return self.fc2(context).squeeze(-1)


rnn_model = RNNModel(input_size=N_INPUT).to(DEVICE)
print(rnn_model)

model_name_rnn = 'best_rnn_attention'
models[model_name_rnn] = train_model(rnn_model, model_name_rnn, n_neg / max(n_pos, 1))

In [110]:
def show_model_metrics_optimized(model, model_name, model_path):
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()

    test_probs, test_true = [], []
    with torch.no_grad():
        for X_b, y_b in test_loader:
            logits = model(X_b.to(DEVICE))
            test_probs.extend(torch.sigmoid(logits).cpu().numpy())
            test_true.extend(y_b.numpy())

    test_probs = np.array(test_probs)
    test_true = np.array(test_true)

    # Cálculo automático del umbral óptimo (Youden's J statistic)
    fpr, tpr, thresholds = roc_curve(test_true, test_probs)
    best_thresh = thresholds[np.argmax(tpr - fpr)]

    preds = (test_probs >= best_thresh).astype(int)

    print(f'=== LSTM {model_name} (Umbral Opt: {best_thresh:.3f}) ===')
    print(f'AUROC: {roc_auc_score(test_true, test_probs):.4f}')
    print(classification_report(test_true, preds, target_names=['Control', 'Sepsis']))

for model_name, values in models.items():
    show_model_metrics_optimized(
        model=values['model'],
        model_name=model_name,
        model_path=values['path'],
    )

=== LSTM best_lstm_bidirectional (Umbral Opt: 0.506) ===
AUROC: 0.7402
              precision    recall  f1-score   support

     Control       0.64      0.63      0.64      1360
      Sepsis       0.73      0.73      0.73      1847

    accuracy                           0.69      3207
   macro avg       0.68      0.68      0.68      3207
weighted avg       0.69      0.69      0.69      3207

